# singular-vectors

In [ ]:
import functools
from collections.abc import Sequence
from pathlib import Path

import seaborn as sns
import xarray as xr
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from matplotlib.axes import Axes

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import CrossDecomposition
from lib.utilities import JOURNAL_MATPLOTLIBRC


def plot_row_of_singular_vectors(
    *,
    ax: Axes,
    ranks: Sequence[int],
    subject: int,
    singular_vectors: xr.DataArray,
    **kwargs,
) -> None:
    inset_axes = nsd.plot_multiple_brain_maps(
        ax=ax,
        maps=[singular_vectors.sel(component=rank) for rank in reversed(ranks)],
        subject=subject,
        **kwargs,
    )
    for rank, ax_ in zip(reversed(ranks), inset_axes, strict=True):
        ax_.set_title(f"{rank}", y=0.9)


FIGURES_HOME = Path.cwd().parent / "figures"

REFERENCE_SUBJECT = 0

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

## load dataset

In [ ]:
dataset = nsd.load_dataset(subject=REFERENCE_SUBJECT, roi="general")
datasets = split_by_repetition(
    filter_by_stimulus(
        dataset,
        stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
    ),
    n_repetitions=2,
)
cross_decomposition = CrossDecomposition(randomized=True)
cross_decomposition.fit(datasets[0], datasets[1])

singular_vectors = cross_decomposition.singular_vectors(direction="left").set_xindex([
    "x",
    "y",
    "z",
])

## plot singular-vectors

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.axis("off")

func = functools.partial(
    plot_row_of_singular_vectors,
    ax=ax,
    singular_vectors=singular_vectors,
    subject=REFERENCE_SUBJECT,
)
func(
    ranks=[1, 2, 3, 4, 5, 10],
    bottom=0.5,
)
func(
    ranks=[20, 50, 100, 500, 1000, 5000],
    bottom=0.05,
)

save_figure(
    fig,
    filepath=FIGURES_HOME / "singular-vectors.pdf",
    dpi=300,
)